<a href="https://colab.research.google.com/github/glebas2310/Pioneer_2_Mini/blob/main/pt_to_rknn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ПЕРЕД ВЫПОЛНЕНИЕМ ПРОГРАММЫ: СРЕДА ВЫПОЛНЕНИЯ -> СМЕНИТЬ СРЕДУ ВЫПОЛНЕНИЯ -> ГРАФИЧЕСКИЙ ПРОЦЕССОР Т4

1. Подготовка и установка библиотек

In [ ]:
!pip install ultralytics roboflow -q
import os
print("✅ Базовые библиотеки установлены!")

2. Загрузга датасета.

---

---




RoboFlow: Versions -> Download dataset -> YOLOv8 -> Show download code -> Jupyter -> Copy


---


---
Вставьте код между горизонтальными линиями в коде.


In [ ]:
from roboflow import Roboflow

# ==============================================================
#                    СЮДА КОД НА УСТАНОВКУ
# ==============================================================

dataset_yaml = f"{dataset.location}/data.yaml"
print(f"✅ Датасет готов! Путь: {dataset_yaml}")

3. Обучение best.pt и автоматический экспорт в best.onnx

In [ ]:
from ultralytics import YOLO

print("🚀 Начинаем обучение YOLO11n...")
model = YOLO('yolo11n.pt')

results = model.train(data=dataset_yaml, epochs=50, imgsz=640, batch=16, plots=True)

best_pt_path = str(results.save_dir / 'weights' / 'best.pt')
print(f"✅ Модель обучена! Веса сохранены в: {best_pt_path}")

print("⚙️ Экспорт в ONNX для NPU...")
best_model = YOLO(best_pt_path)
onnx_path = best_model.export(format='onnx', opset=12, simplify=True)
print(f"✅ ONNX файл готов: {onnx_path}")

4. Установка RKNN-Toolkit2

In [ ]:
%%bash
# ЯЧЕЙКА 4: Создание среды для компилятора NPU
echo "Установка менеджера uv..."
pip install uv -q

echo "Очистка старых сред и создание новой..."
rm -rf env310
uv python install 3.10
uv venv env310 --python 3.10 --seed

echo "Установка rknn-toolkit2 и зависимостей..."
# Строго фиксируем setuptools==69.5.1, чтобы вернуть модуль pkg_resources
./env310/bin/pip install "numpy<2.0.0" setuptools==69.5.1 onnx==1.16.1 rknn-toolkit2 -q

echo "✅ Среда для RKNN успешно создана!"

5. Конвертация из .onnx в .rknn

In [ ]:
# ЯЧЕЙКА 5: Сборка RKNN-модели под RK3576
output_rknn = "/content/best_model.rknn"

convert_script = f"""
from rknn.api import RKNN

rknn = RKNN(verbose=False)
# Указываем реальный чип дрона — rk3576
rknn.config(mean_values=[[0, 0, 0]], std_values=[[255, 255, 255]], target_platform="rk3576")

print("Загрузка ONNX: {onnx_path}")
ret = rknn.load_onnx(model="{onnx_path}")
if ret != 0: exit(ret)

print("Сборка графа под RK3576...")
ret = rknn.build(do_quantization=False)
if ret != 0: exit(ret)

print("Сохранение модели...")
ret = rknn.export_rknn("{output_rknn}")
if ret != 0: exit(ret)

rknn.release()
"""

with open("convert.py", "w") as f:
    f.write(convert_script)

!./env310/bin/python convert.py